# Docflow — dots.mocr inference server (Colab dev tool)

Runs `dots.ocr` on Colab's GPU and exposes a `/parse` endpoint that the local Docflow
engine calls via `RemoteDotsModel`. This is a **throwaway development tool** for Phase 0 —
production inference becomes a Modal function in Phase 1.

**Usage**
1. Runtime → Change runtime type → **GPU** (T4 is enough).
2. Run every cell top to bottom.
3. The last cell prints a public URL. On your machine set it and run the CLI:
   - PowerShell: `$env:DOCFLOW_DOTS_URL="<url>"`
   - then: `uv run docflow convert file.pdf --model dots -o out/`

**Debugging:** if `/parse` returns 500, the server puts the full traceback in the JSON
response (the local CLI prints it). You can also run the **sanity-check cell** below to
test inference directly. The server cell is **re-runnable** — fix a cell and re-run it
without restarting the runtime or the tunnel.

Response schema matches Docflow's `PageLayout` contract:
`{page_index, image_width, image_height, elements: [{category, bbox, text}]}`.

In [ ]:
# 1. Dependencies
!pip install -q -U transformers accelerate qwen_vl_utils huggingface_hub flask flask-cors pyngrok pillow

In [ ]:
# 2. Download weights to a dot-free folder.
# The HF repo id 'rednote-hilab/dots.ocr' contains a '.', which breaks transformers'
# trust_remote_code dynamic import. Downloading to ./DotsOCR sidesteps that.
from huggingface_hub import snapshot_download

MODEL_DIR = snapshot_download(repo_id="rednote-hilab/dots.ocr", local_dir="./DotsOCR")
print("weights at:", MODEL_DIR)

In [ ]:
# 3. Load the model and processor. If this cell errors, STOP and fix it here —
# a failed load is the most common cause of a later 500 from /parse.
import torch
from transformers import AutoModelForCausalLM, AutoProcessor

model = AutoModelForCausalLM.from_pretrained(
    "./DotsOCR",
    attn_implementation="sdpa",  # avoids a flash-attn build on Colab
    torch_dtype=torch.bfloat16,
    device_map="auto",
    trust_remote_code=True,
)
processor = AutoProcessor.from_pretrained("./DotsOCR", trust_remote_code=True)
print("model loaded on:", model.device)

In [ ]:
# 4. Inference: one PIL image -> list of layout-element dicts.
import json
import re
from qwen_vl_utils import process_vision_info

# Faithful rendering of dots.ocr's prompt_layout_all_en. If you have the official
# dots_ocr package, replace this with its exact prompt string for best results.
PROMPT = (
    "Please output the layout information from this image, including each layout "
    "element's bbox, its category, and the corresponding text content.\n\n"
    "1. Bbox format: [x1, y1, x2, y2] in pixels of this image.\n"
    "2. Categories: ['Caption','Footnote','Formula','List-item','Page-footer',"
    "'Page-header','Picture','Section-header','Table','Text','Title'].\n"
    "3. Text rules: Picture -> omit text; Formula -> LaTeX; Table -> HTML; others -> Markdown.\n"
    "4. Output a single JSON array of elements, sorted in human reading order."
)


def _to_elements(text):
    """Parse the model's decoded text into a list of element dicts.
    Tolerates code fences and either a JSON array or a {\"elements\": [...]} object."""
    text = text.strip()
    if text.startswith("```"):
        text = re.sub(r"^```[a-zA-Z]*\n?|\n?```$", "", text).strip()
    try:
        data = json.loads(text)
    except json.JSONDecodeError:
        m = re.search(r"(\[.*\]|\{.*\})", text, re.DOTALL)
        if not m:
            raise ValueError(f"No JSON found in model output: {text[:500]!r}")
        data = json.loads(m.group(0))
    if isinstance(data, dict):
        for key in ("elements", "layout", "results"):
            if isinstance(data.get(key), list):
                return data[key]
        return [data]
    return data


def parse_image(image):
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": image},
                {"type": "text", "text": PROMPT},
            ],
        }
    ]
    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    image_inputs, video_inputs = process_vision_info(messages)
    inputs = processor(
        text=[text], images=image_inputs, videos=video_inputs,
        padding=True, return_tensors="pt",
    ).to(model.device)

    generated = model.generate(**inputs, max_new_tokens=16384)
    trimmed = [out[len(inp):] for inp, out in zip(inputs.input_ids, generated)]
    decoded = processor.batch_decode(trimmed, skip_special_tokens=True)[0]
    return _to_elements(decoded)

In [ ]:
# 4b. (Optional) Sanity-check inference directly — surfaces any error with a full
# traceback, without going through HTTP. Replace the blank image with a real page
# image for a meaningful result.
import traceback
from PIL import Image

try:
    out = parse_image(Image.new("RGB", (1000, 1400), "white"))
    print("OK -", type(out), "| first elements:", out[:2] if isinstance(out, list) else out)
except Exception:
    traceback.print_exc()

In [ ]:
# 5. Flask server exposing /parse and /health. Re-runnable: it shuts down any
# server started by a previous run of this cell before starting a new one.
import io
import threading
import traceback
from flask import Flask, jsonify, request
from flask_cors import CORS
from PIL import Image
from werkzeug.serving import make_server

try:
    _server.shutdown()  # noqa: F821 -- defined on a previous run
except NameError:
    pass

app = Flask(__name__)
CORS(app)


@app.get("/health")
def health():
    return jsonify(status="ok")


@app.post("/parse")
def parse():
    try:
        page_index = int(request.form.get("page_index", 0))
        image = Image.open(io.BytesIO(request.files["image"].read())).convert("RGB")
        elements = parse_image(image)
        norm = [
            {"category": e.get("category"), "bbox": e.get("bbox"), "text": e.get("text")}
            for e in elements
            if isinstance(e, dict)
        ]
        return jsonify(
            page_index=page_index,
            image_width=image.width,
            image_height=image.height,
            elements=norm,
        )
    except Exception as exc:
        tb = traceback.format_exc()
        print("ERROR in /parse:\n", tb)
        return jsonify(error=str(exc), traceback=tb), 500


_server = make_server("0.0.0.0", 5000, app, threaded=True)
threading.Thread(target=_server.serve_forever, daemon=True).start()
print("Flask serving on :5000 (re-runnable)")

## 6. Expose a public URL

Uses **pyngrok**. Get a free authtoken at <https://dashboard.ngrok.com> and paste it below.
You only need to run this **once** per session — if you re-run the server cell above, the
tunnel keeps pointing at port 5000.

In [ ]:
from pyngrok import ngrok

ngrok.set_auth_token("PASTE_YOUR_NGROK_AUTHTOKEN_HERE")
public_url = ngrok.connect(5000).public_url
print("DOCFLOW_DOTS_URL =", public_url)
print("\nOn your machine (PowerShell):")
print(f'  $env:DOCFLOW_DOTS_URL="{public_url}"')
print("  uv run docflow convert file.pdf --model dots -o out/")